# Sentinel-2 Satellite Time Series (.pt) Decoder & Ground Truth Visualizer
### High-Fidelity Decoding & Verification on Representative Agricultural Patches
> **Dataset:** `minh279/sentinel2-crop-tsvit-dinov3-2022`  
> **Key Objectives:**
> 1. Decode PyTorch dictionary `{'image', 'doy', 'label'}` from raw `.pt` tensor files.
> 2. Restore true bottom-of-atmosphere (BOA) surface reflectance via de-normalization.
> 3. Preserve crisp 10m physical pixel boundaries (`nearest` interpolation) and dynamic contrast stretch.
> 4. Auto-select clearest acquisition frame (`select_best_timestamp`).
> 5. Verify 1:1 spatial alignment between satellite field parcels and official French RPG 2022 Ground Truth.

In [ ]:
import os
import glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch

# Fast rendering settings
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.sans-serif'] = ['DejaVu Sans', 'Arial', 'sans-serif']

print(f"PyTorch Version: {torch.__version__} | Device: {'CUDA' if torch.cuda.is_available() else 'CPU'}")

In [ ]:
# PASTIS-24 normalization parameters (10 Sentinel-2 bands: B02 to B12)
S2_MEAN_10B = torch.tensor([
    1165.94, 1375.65, 1429.22, 1764.80, 2719.27,
    3063.61, 3205.90, 3319.11, 2422.90, 1639.37
], dtype=torch.float32).view(1, 10, 1, 1)

S2_STD_10B = torch.tensor([
    1942.62, 1881.92, 1959.38, 1867.22, 1754.59,
    1769.40, 1784.86, 1767.71, 1458.96, 1299.28
], dtype=torch.float32).view(1, 10, 1, 1)

# 19 French RPG 2022 crop categories
CROP_CLASSES = {
    0: "Non-agri / Road", 1: "Permanent meadow", 2: "Soft winter wheat",
    3: "Corn / Maize", 4: "Winter barley", 5: "Winter rapeseed",
    6: "Spring barley", 7: "Sunflower", 8: "Soybean",
    9: "Peas", 10: "Protein crops", 11: "Other oilseeds",
    12: "Vegetables / Flowers", 13: "Vineyard", 14: "Fruit orchards (Apples)",
    15: "Fallow", 16: "Potato", 17: "Sugar beet", 18: "Other cereals"
}

# Categorical color palette for clear distinction
CROP_COLORS = [
    '#343a40', '#2ca02c', '#f5b041', '#e67e22', '#d68910',
    '#27ae60', '#f39c12', '#f1c40f', '#1abc9c', '#16a085',
    '#3498db', '#2980b9', '#10b981', '#8e44ad', '#9b59b6',
    '#a04000', '#d35400', '#e74c3c', '#8000ff'
]

CMAP_CROP = ListedColormap(CROP_COLORS)
NORM_CROP = BoundaryNorm(boundaries=np.arange(len(CROP_COLORS) + 1) - 0.5, ncolors=len(CROP_COLORS))

In [ ]:
def locate_dataset():
    """Fast dataset locator for Kaggle and local environments."""
    # 1. Search Kaggle input directory
    if os.path.exists("/kaggle/input"):
        for root, _, files in os.walk("/kaggle/input"):
            pt_files = [f for f in files if f.endswith(".pt")]
            if pt_files:
                print(f"Kaggle: Found {len(pt_files):,} patches in {root}")
                return {f: os.path.join(root, f) for f in pt_files}

    # 2. Search local workspace paths
    for base in ["./crop_classification_tsvit_dinov3/data/processed/patches", "./data/processed/patches", "."]:
        if os.path.exists(base):
            pts = glob.glob(os.path.join(base, "**/*.pt"), recursive=True)
            if pts:
                print(f"Local: Found {len(pts):,} patches in {base}")
                return {os.path.basename(p): p for p in pts}

    raise FileNotFoundError("No .pt patch files found!")

PATCH_MAP = locate_dataset()

## 1. Decode PyTorch File Structure (`patch_00233.pt`)
Each `.pt` file contains:
- `image`: Shape `[25, 10, 24, 24]` $\rightarrow$ 25 dates, 10 spectral bands, 24x24 px (10m GSD $= 240\text{m} \times 240\text{m}$).
- `doy`: Shape `[25]` $\rightarrow$ Normalized Day of Year ($DOY / 365.0$).
- `label`: Shape `[24, 24]` $\rightarrow$ Pixel-wise RPG 2022 crop category IDs.

In [ ]:
target_key = 'patch_00233.pt' if 'patch_00233.pt' in PATCH_MAP else list(PATCH_MAP.keys())[0]
sample = torch.load(PATCH_MAP[target_key], map_location='cpu', weights_only=False)

print(f"Decoded File: {target_key}")
print(f"- Image Tensor: {list(sample['image'].shape)} [T=25, C=10, H=24, W=24]")
print(f"- DOY Tensor:   {list(sample['doy'].shape)} [Day of Year]")
print(f"- Label Tensor: {list(sample['label'].shape)} [Pixel-wise class IDs]")

labels, counts = torch.unique(sample['label'], return_counts=True)
print("\nGround Truth Class Distribution:")
for c, cnt in zip(labels.tolist(), counts.tolist()):
    pct = (cnt / sample['label'].numel()) * 100
    print(f"  * Class {c:02d} ({CROP_CLASSES[c]}): {cnt} px ({pct:.1f}%)")

In [ ]:
def denormalize(tensor_10b):
    """Reconstruct BOA surface reflectance: raw = norm * std + mean."""
    return tensor_10b * S2_STD_10B + S2_MEAN_10B


def select_best_timestamp(raw_cube):
    """Find clearest acquisition timestamp with highest vegetation contrast."""
    scores = []
    for t in range(raw_cube.shape[0]):
        nir = raw_cube[t, 6]
        red = raw_cube[t, 2]
        blue = raw_cube[t, 0]
        # Skip cloud oversaturation or corrupted frames
        if red.mean() < 150 or red.mean() > 4500 or blue.mean() > 3500:
            scores.append(-999)
            continue
        ndvi = (nir - red) / (nir + red + 1e-6)
        scores.append(ndvi.mean().item() * 0.7 + (raw_cube[t, [2, 1, 0]].std().item() / 1000.0) * 0.3)
    return int(np.argmax(scores))


def stretch_contrast(arr_3ch):
    """Percentile clip (2%-98%) + gamma 1.1 for sharp natural color rendering."""
    p2, p98 = np.percentile(arr_3ch, (2, 98))
    res = np.clip((arr_3ch - p2) / (p98 - p2 + 1e-6), 0.0, 1.0)
    return np.power(res, 1.0 / 1.1)

## 2. Primary Showcase: `patch_00233.pt` (Sharp 10m Pixels & Boundaries)
Using `nearest` interpolation guarantees crisp, un-blurred 10m pixel boundaries matching ground parcel geometry.

In [ ]:
d = torch.load(PATCH_MAP[target_key], map_location='cpu', weights_only=False)
raw = denormalize(d['image'])
t_best = select_best_timestamp(raw)
doy = int(round(d['doy'][t_best].item() * 365.0))

rgb = stretch_contrast(raw[t_best, [2, 1, 0]].permute(1, 2, 0).numpy())
cir = stretch_contrast(raw[t_best, [6, 2, 1]].permute(1, 2, 0).numpy())
lbl = d['label'].numpy()
classes = np.unique(lbl).tolist()

fig, axes = plt.subplots(1, 4, figsize=(15, 3.8), facecolor='white')

# 1. True Color RGB
axes[0].imshow(rgb, interpolation='nearest')
axes[0].set_title(f"True Color RGB (10m)\nDOY {doy} (Frame #{t_best+1})", fontsize=10, fontweight='bold')
axes[0].axis('off')

# 2. False Color CIR (NIR)
axes[1].imshow(cir, interpolation='nearest')
axes[1].set_title("False Color CIR (NIR)\n[Red = Healthy Canopy]", fontsize=10, fontweight='bold')
axes[1].axis('off')

# 3. Ground Truth Mask
axes[2].imshow(lbl, cmap=CMAP_CROP, norm=NORM_CROP, interpolation='nearest')
axes[2].set_title("Ground Truth Mask\n[RPG 2022 Labels]", fontsize=10, fontweight='bold')
axes[2].axis('off')

# 4. Superimposed Overlay
axes[3].imshow(rgb, interpolation='nearest')
axes[3].imshow(lbl, cmap=CMAP_CROP, norm=NORM_CROP, alpha=0.45, interpolation='nearest')
if len(classes) > 1:
    axes[3].contour(lbl, levels=classes, colors='black', linewidths=1.8)
axes[3].set_title("Overlay: RGB + GT\n[Black Line = Field Boundary]", fontsize=10, fontweight='bold')
axes[3].axis('off')

handles = [Patch(facecolor=CROP_COLORS[c], edgecolor='black', label=f"[{c}] {CROP_CLASSES[c]}") for c in classes]
axes[3].legend(handles=handles, loc='upper right', bbox_to_anchor=(1.75, 1.0), fontsize=9)

plt.suptitle(f"Primary Showcase: {target_key}", fontsize=12, fontweight='bold', y=1.03)
plt.tight_layout()
plt.show()

## 3. Gallery of 10 Representative Agricultural Patches
Evaluating 10 diverse agricultural patches across different crop types, proving 1:1 ground truth field alignment.

In [ ]:
SELECTED_PATCHES = [
    'patch_00233.pt', 'patch_00583.pt', 'patch_00597.pt', 'patch_00013.pt', 'patch_00028.pt',
    'patch_00027.pt', 'patch_00012.pt', 'patch_00387.pt', 'patch_00401.pt', 'patch_00779.pt'
]

valid_list = [p for p in SELECTED_PATCHES if p in PATCH_MAP]
if len(valid_list) < 10:
    for k in PATCH_MAP:
        if k not in valid_list and len(valid_list) < 10:
            valid_list.append(k)

fig, axes = plt.subplots(len(valid_list), 4, figsize=(14, 2.8 * len(valid_list)), facecolor='white')

for i, fname in enumerate(valid_list):
    sample_data = torch.load(PATCH_MAP[fname], map_location='cpu', weights_only=False)
    cube = denormalize(sample_data['image'])
    t_opt = select_best_timestamp(cube)
    doy_val = int(round(sample_data['doy'][t_opt].item() * 365.0))

    rgb_img = stretch_contrast(cube[t_opt, [2, 1, 0]].permute(1, 2, 0).numpy())
    cir_img = stretch_contrast(cube[t_opt, [6, 2, 1]].permute(1, 2, 0).numpy())
    gt_mask = sample_data['label'].numpy()
    present_cls = np.unique(gt_mask).tolist()
    cls_labels = ", ".join([f"[{c}] {CROP_CLASSES[c].split(' / ')[0]}" for c in present_cls])

    # Col 0: RGB
    axes[i, 0].imshow(rgb_img, interpolation='nearest')
    axes[i, 0].set_title(f"#{i+1}: {fname} (DOY {doy_val})\nTrue Color (RGB 10m)", fontsize=9, fontweight='bold')
    axes[i, 0].axis('off')

    # Col 1: CIR
    axes[i, 1].imshow(cir_img, interpolation='nearest')
    axes[i, 1].set_title("False Color (CIR NIR)", fontsize=9, fontweight='bold')
    axes[i, 1].axis('off')

    # Col 2: Ground Truth
    axes[i, 2].imshow(gt_mask, cmap=CMAP_CROP, norm=NORM_CROP, interpolation='nearest')
    axes[i, 2].set_title(f"Ground Truth\n{cls_labels}", fontsize=8.5, fontweight='bold')
    axes[i, 2].axis('off')

    # Col 3: Overlay + Boundary
    axes[i, 3].imshow(rgb_img, interpolation='nearest')
    axes[i, 3].imshow(gt_mask, cmap=CMAP_CROP, norm=NORM_CROP, alpha=0.45, interpolation='nearest')
    if len(present_cls) > 1:
        axes[i, 3].contour(gt_mask, levels=present_cls, colors='black', linewidths=1.2)
    axes[i, 3].set_title("Overlay (Field Boundaries)", fontsize=9, fontweight='bold')
    axes[i, 3].axis('off')

plt.tight_layout()
plt.show()